# Monitoring DSPy with MLflow

This notebook accompanies Chapter 10 §10.1 of *Context Engineering with DSPy*: **Monitoring and Observability with MLflow**. Three lines of setup give you full tracing of every DSPy call. Another three give you optimizer tracking with parent-child run hierarchy, so you can compare GEPA trials side by side and rescue trace data when an overnight optimization crashes.

**Required environment variables** (loaded from `.env`):

- `OPENAI_API_KEY`

**Service dependencies**: a local MLflow server. Start it in a separate terminal before running the cells below:

```bash
mlflow server --backend-store-uri sqlite:///mydb.sqlite
```

MLflow then serves its UI at <http://127.0.0.1:5000>.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5.6-luna'))

## 10.1.1 Setting up MLflow

Point DSPy at your local MLflow server and turn on autologging. From now on every DSPy program you run generates traces automatically. The SQLite backend in the server command above keeps things fast once you have more than a few hundred runs — file-based storage gets slow quickly.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("DSPy")
mlflow.dspy.autolog()

## 10.1.2 Viewing logs

Define the `SentimentClassifier` from Chapter 9 and run a single call. Open <http://127.0.0.1:5000>, select the **DSPy** experiment, and navigate to the Traces tab. You'll see a tree view of every step: module call, adapter format, the raw LM request and response, adapter parse, and final output.

In [ ]:
from typing import Literal

class SentimentClassifier(dspy.Signature):
    """Classify sentiment of a given text."""
    text: str = dspy.InputField()
    sentiment: Literal['positive', 'negative', 'neutral'] = dspy.OutputField()

classifier = dspy.ChainOfThought(SentimentClassifier)

result = classifier(text="I love this product, it works perfectly!")
print(result.sentiment)

## 10.1.3 Tracing optimizers

Optimizers like GEPA do a lot under the hood, so the real win shows up when you trace them. Enable the full set of compile-time and eval-time hooks:

In [ ]:
mlflow.dspy.autolog(
    log_compiles=True,
    log_evals=True,
    log_traces_from_compile=True,
)

Now run a GEPA optimization on the `SentimentClassifier`. MLflow records a **parent run** for the whole compile plus a **child run** for every evaluation. Each child stores the candidate program state, score, and full LM traces, so you can compare trials with the final program side by side.

GEPA requires exactly one budget argument (`auto`, `max_metric_calls`, or `max_full_evals`) and a reflection LM. This example uses a light automatic budget and the stronger teacher model for reflection.

In [ ]:
from dspy.evaluate import Evaluate

# Tiny train and dev sets just to demonstrate the trace hierarchy.
train_examples = [
    dspy.Example(text="This is amazing, best purchase ever!", sentiment="positive").with_inputs("text"),
    dspy.Example(text="Completely broken, total waste of money.", sentiment="negative").with_inputs("text"),
    dspy.Example(text="It arrived on time and works as described.", sentiment="neutral").with_inputs("text"),
]
dev_examples = [
    dspy.Example(text="Honestly the highlight of my month.", sentiment="positive").with_inputs("text"),
    dspy.Example(text="Stopped working after one day.", sentiment="negative").with_inputs("text"),
]

def accuracy(example, pred, trace=None, pred_name=None, pred_trace=None):
    return float(pred.sentiment.strip().lower() == example.sentiment.lower())

reflection_lm = dspy.LM("openai/gpt-5.6-sol", temperature=1.0)

optimizer = dspy.GEPA(
    metric=accuracy,
    auto="light",
    reflection_lm=reflection_lm,
)
optimized_classifier = optimizer.compile(
    classifier,
    trainset=train_examples,
    valset=dev_examples,
)

## 10.1.4 Tracking experiments

Beyond optimization, MLflow is great for plain comparisons between models or configurations. The pattern below loops over two models, runs the same evaluation under a named run for each, and logs the score plus the model name as parameters. After a few sweeps you have a clean comparison table in MLflow showing which model wins on *your* task — not the published benchmarks.

In [ ]:
# Compare models on the same task
for model_name in ["openai/gpt-5.6-sol", "openai/gpt-5.6-luna"]:
    lm = dspy.LM(model_name)
    dspy.configure(lm=lm)

    with mlflow.start_run(run_name=model_name):
        evaluator = Evaluate(
            metric=accuracy,
            devset=dev_examples,
            num_threads=4,
        )
        result = evaluator(classifier)
        mlflow.log_metric("sentiment_score", result.score)
        mlflow.log_param("model", model_name)

Log whatever else you need to track alongside accuracy — token counts, dollar cost, p95 latency, eval CSVs. In production, knowing that your program costs $0.03 per call matters just as much as knowing it scores 89% on your eval.

In [ ]:
with mlflow.start_run():
    mlflow.log_metric("token_count", 12_500)
    mlflow.log_metric("cost_usd", 0.03)
    mlflow.log_metric("latency_p95", 1.8)
    # mlflow.log_artifact("evaluation_results.csv")

## 10.1.5 Querying traces with the MLflow MCP server

The UI is fine for one trace at a time. It gets painful the moment you want to ask questions across hundreds of traces — *"which trials in this GEPA run regressed on the bank-details field?"*, *"which trace had the longest latency in the last hour?"* MLflow ships an MCP server that exposes your tracking server's traces to any MCP-compatible coding agent (Claude Code, Codex, Cursor). Install it as an extra on top of MLflow, then register it with Claude Code:

In [ ]:
%pip install 'mlflow[mcp]>=3.5.1' -q

# Register the MLflow MCP server with Claude Code (run in a terminal):
#
#   claude mcp add mlflow-mcp -e MLFLOW_TRACKING_URI=http://127.0.0.1:5000 \
#     -- uv run --with "mlflow[mcp]>=3.5.1" mlflow mcp run

If you'd rather check the config into your repo than register it globally, drop a `.mcp.json` at the project root:

```json
{
  "mcpServers": {
    "mlflow-mcp": {
      "command": "uv",
      "args": ["run", "--with", "mlflow[mcp]>=3.5.1", "mlflow", "mcp", "run"],
      "env": {
        "MLFLOW_TRACKING_URI": "http://127.0.0.1:5000"
      }
    }
  }
}
```

Once connected, the agent can call ten trace operations — `search_traces`, `get_trace`, `delete_traces`, `set_trace_tag`, `delete_trace_tag`, `log_feedback`, `log_expectation`, `get_assessment`, `update_assessment`, and `delete_assessment`. The two we reach for most are `search_traces` for triage and `log_feedback` for human-in-the-loop labeling — once a teammate flags a trace as a bad output in chat, the agent writes that judgment back to the trace as ground truth, and it becomes training data for your next GEPA run.

One practical note: pass `extract_fields` to `search_traces` (a comma-separated list of dot paths like `info.trace_id,info.state,data.spans.*.name`) to keep response sizes small. The default response carries the full span tree for every matching trace and blows out your context window after a handful of results. You can also scope which tools are exposed with the `MLFLOW_MCP_TOOLS` environment variable (`genai`, `ml`, `all`, or a comma-separated list).